<h1 align="center">TÉCNICO EM CIÊNCIA DE DADOS</h1>

<h2 align="center">Colocando em prática — Auditoria final de uma base preparada para análise</h2>

**Componente:** Análise Exploratória de Dados e Inteligência de Negócios  
**Unidade Curricular:** Análise Exploratória de Dados  
**Tema da Semana:** Transformação e Preparação de Dados  
**Semana 9 — Aula 4:** Análise da adequação dos dados preparados

# Situação profissional

Uma empresa brasileira de serviços técnicos prepara um relatório operacional para comparar **canais de atendimento, unidades, tipos de serviço, valores de serviços e produtividade das equipes**.

Um analista realizou uma primeira preparação da base e registrou transformações que pretende aplicar. Antes de liberar os dados para novas análises, sua dupla fará uma **auditoria das decisões de preparação**.

## Missão

> **Analisar se cada transformação proposta é adequada ao objetivo da análise, comparando os dados originais com os efeitos da transformação e documentando a decisão.**

Para cada proposta, a decisão profissional será uma destas:

- **Aplicar** — a transformação atende à necessidade;
- **Revisar** — a proposta precisa ser alterada antes de ser usada;
- **Não transformar** — a variável já atende à necessidade e a transformação proposta não acrescenta utilidade.

**Importante:** registrem a decisão no roteiro **antes de executar a célula de conferência de cada proposta**. O notebook não usa `input()` e nenhuma resposta de vocês bloqueia a execução.

**Dados:** simulados para fins didáticos; não representam uma empresa real.  
**Tempo previsto:** 43 minutos.

# Direção investigativa da atividade

**Pergunta:** as transformações propostas tornam a base adequada às comparações previstas sem distorcer, eliminar ou inventar informação necessária?

**Avanço esperado:** avaliar as propostas com base em coerência com o objetivo, preservação de significado, verificação do resultado e documentação.

**Produto:** registro profissional das decisões de **Aplicar / Revisar / Não transformar** e uma base preparada para novas explorações.

**Uso posterior:** a base somente deve ser liberada quando as transformações aprovadas estiverem justificadas e verificadas.

## Preparação

Este notebook é autossuficiente: a base simulada é criada na própria atividade.

Execute as células **na ordem em que aparecem**. O código serve para mostrar os efeitos das propostas; o foco é **analisar a adequação das decisões**, não programar as transformações.

In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# Base simulada: cada registro representa uma equipe em um dia de operação.
dados_originais = pd.DataFrame({
    "registro": [f"R{i:02d}" for i in range(1, 13)],
    "data": pd.to_datetime([
        "2026-09-01","2026-09-01","2026-09-01","2026-09-02",
        "2026-09-02","2026-09-02","2026-09-03","2026-09-03",
        "2026-09-03","2026-09-04","2026-09-04","2026-09-04"
    ]),
    "unidade": [
        "Campinas","Sorocaba","Jundiaí","Campinas",
        "Sorocaba","Jundiaí","Campinas","Sorocaba",
        "Jundiaí","Campinas","Sorocaba","Jundiaí"
    ],
    "canal_atendimento": [
        "WhatsApp"," whatsapp ","WHATSAPP","Telefone",
        "telefone","E-mail","e-mail","WhatsApp",
        "Telefone"," WHATSAPP ","E-mail","telefone"
    ],
    "tipo_servico": [
        "Instalação","Manutenção","Reparo","Manutenção",
        "Instalação","Reparo","Instalação","Manutenção",
        "Reparo","Manutenção","Instalação","Reparo"
    ],
    "ordens_concluidas": [8, 10, 6, 12, 9, 11, 7, 13, 8, 10, 14, 9],
    "horas_tecnicas": [4.0, 4.0, 3.0, 5.0, 4.5, 5.0, 3.5, 5.0, 4.0, 4.0, 6.0, 4.5],
    "valor_servicos": [
        "1250.50","1680.00","980.75","2140.00",
        "1525.20","1875.50","1160.00","2255.80",
        "1340.25","1760.40","2480.00","1495.60"
    ]
})

print("Base original")
display(dados_originais)
print("\nTipos reconhecidos pelo pandas")
print(dados_originais.dtypes)

# Objetivos do relatório e plano proposto

O relatório que será produzido depois desta preparação precisa:

- comparar os **canais de atendimento**;
- comparar as **unidades**, mantendo sua identificação;
- comparar separadamente **Instalação, Manutenção e Reparo**;
- realizar cálculos com `valor_servicos`;
- comparar o volume de ordens considerando as `horas_tecnicas` disponíveis.

O analista propôs as transformações abaixo.

In [ ]:
plano_proposto = pd.DataFrame({
    "variavel": [
        "canal_atendimento",
        "unidade",
        "tipo_servico",
        "valor_servicos",
        "ordens_concluidas + horas_tecnicas"
    ],
    "proposta": [
        "Padronizar rótulos equivalentes",
        "Substituir os nomes por códigos 1, 2 e 3",
        "Agrupar Instalação e Manutenção em uma única categoria",
        "Converter de texto para numérico",
        "Criar ordens_por_hora_tecnica"
    ],
    "finalidade_declarada": [
        "Comparar canais",
        "Facilitar a análise",
        "Simplificar os dados",
        "Realizar cálculos",
        "Comparar produtividade"
    ]
})

display(plano_proposto)

## Etapa 1 — Avaliação inicial

Antes de executar as células seguintes, preencham no roteiro uma decisão preliminar para **cada uma das cinco propostas**:

**Aplicar | Revisar | Não transformar**

A justificativa deve considerar o objetivo do relatório e o significado da variável. Depois disso, prossigam para examinar os efeitos de cada proposta.

# Proposta 1 — `canal_atendimento`

A proposta é **padronizar rótulos equivalentes** para comparar os canais.

Antes de executar, registrem no roteiro sua decisão e a justificativa. Em seguida, observem o que a transformação faz com os registros.

In [ ]:
def padronizar_canal(valor):
    chave = valor.strip().casefold()
    mapa = {
        "whatsapp": "WhatsApp",
        "telefone": "Telefone",
        "e-mail": "E-mail",
        "email": "E-mail",
    }
    return mapa.get(chave, valor.strip())

canal_padronizado = dados_originais["canal_atendimento"].map(padronizar_canal)

comparacao_canal = pd.DataFrame({
    "antes": dados_originais["canal_atendimento"],
    "depois": canal_padronizado
})

display(comparacao_canal.drop_duplicates())
print("Categorias distintas antes:", dados_originais["canal_atendimento"].nunique())
print("Categorias distintas depois:", canal_padronizado.nunique())

### Registre a verificação

No roteiro, indique:

- se a proposta atende ao objetivo de comparar canais;
- que evidência do antes/depois sustenta sua decisão;
- se o significado das categorias foi preservado.

# Proposta 2 — `unidade`

Os nomes das unidades já aparecem como **Campinas, Sorocaba e Jundiaí**. A proposta é substituí-los por `1`, `2` e `3` para “facilitar a análise”.

Registrem primeiro sua decisão. Depois, executem a simulação da proposta.

In [ ]:
mapa_unidade = {"Campinas": 1, "Sorocaba": 2, "Jundiaí": 3}

simulacao_unidade = dados_originais[["registro", "unidade"]].copy()
simulacao_unidade["unidade_codigo"] = simulacao_unidade["unidade"].map(mapa_unidade)

display(simulacao_unidade)
print("Unidades originais:", sorted(dados_originais["unidade"].unique()))
print("Códigos propostos:", sorted(simulacao_unidade["unidade_codigo"].unique()))

### Registre a verificação

No roteiro, avaliem:

- se os códigos acrescentam informação necessária para comparar as unidades;
- se `1`, `2` e `3` podem sugerir uma ordem que não existe no significado de `unidade`;
- se a transformação deve ser aplicada, revista ou descartada.

# Proposta 3 — `tipo_servico`

O relatório precisa comparar separadamente **Instalação, Manutenção e Reparo**.

A proposta do analista é reunir **Instalação e Manutenção** em uma única categoria e manter Reparo separado.

Registrem primeiro sua avaliação. Depois, observem o efeito da recategorização proposta.

In [ ]:
mapa_servico = {
    "Instalação": "Instalação/Manutenção",
    "Manutenção": "Instalação/Manutenção",
    "Reparo": "Reparo"
}

simulacao_servico = dados_originais[["registro", "tipo_servico"]].copy()
simulacao_servico["tipo_servico_proposto"] = simulacao_servico["tipo_servico"].map(mapa_servico)

display(simulacao_servico)
print("Categorias antes:", sorted(dados_originais["tipo_servico"].unique()))
print("Categorias depois:", sorted(simulacao_servico["tipo_servico_proposto"].unique()))

### Registre a verificação

No roteiro, respondam:

- qual distinção deixa de estar disponível depois do agrupamento;
- se essa perda é compatível com o objetivo do relatório;
- qual decisão profissional é adequada para a proposta.

# Proposta 4 — `valor_servicos`

Os valores estão padronizados visualmente, mas `valor_servicos` ainda é reconhecido como **texto**. O relatório precisará realizar cálculos com essa variável.

Registrem a decisão antes de executar a conversão.

In [ ]:
valor_numerico = pd.to_numeric(dados_originais["valor_servicos"], errors="raise")

comparacao_valor = pd.DataFrame({
    "antes_texto": dados_originais["valor_servicos"],
    "depois_numerico": valor_numerico
})

display(comparacao_valor.head(8))
print("Tipo antes:", dados_originais["valor_servicos"].dtype)
print("Tipo depois:", valor_numerico.dtype)
print("Quantidade de valores preenchidos antes:", dados_originais["valor_servicos"].notna().sum())
print("Quantidade de valores preenchidos depois:", valor_numerico.notna().sum())

### Registre a verificação

No roteiro, indiquem:

- se a conversão é coerente com o uso pretendido;
- o que mudou na representação computacional;
- que evidência mostra que os valores preenchidos foram preservados.

# Proposta 5 — criar `ordens_por_hora_tecnica`

A empresa quer comparar o volume de ordens concluídas considerando as horas técnicas disponíveis.

A proposta é criar:

`ordens_por_hora_tecnica = ordens_concluidas ÷ horas_tecnicas`

Antes de executar, avaliem se a nova variável acrescenta uma informação coerente com essa necessidade.

In [ ]:
if dados_originais["horas_tecnicas"].isna().any():
    raise ValueError("Há valores ausentes em horas_tecnicas.")
if (dados_originais["horas_tecnicas"] <= 0).any():
    raise ValueError("Há valores não positivos em horas_tecnicas.")

ordens_por_hora = dados_originais["ordens_concluidas"] / dados_originais["horas_tecnicas"]

comparacao_produtividade = dados_originais[[
    "registro", "ordens_concluidas", "horas_tecnicas"
]].copy()
comparacao_produtividade["ordens_por_hora_tecnica"] = ordens_por_hora.round(2)

display(comparacao_produtividade)

### Registre a verificação

No roteiro, indiquem:

- que informação a nova variável acrescenta;
- se o denominador é adequado e foi verificado;
- se a transformação contribui para a comparação pretendida.

# Etapa 2 — Auditoria final da base preparada

Agora comparem as decisões registradas com uma **base preparada de referência**, construída apenas com as transformações que preservam as informações necessárias ao relatório.

Avaliem o resultado usando quatro critérios:

1. **Coerência com o objetivo** — a transformação ajuda a responder à análise?
2. **Significado preservado** — alguma informação necessária foi distorcida ou eliminada?
3. **Resultado verificado** — o efeito observado corresponde ao que se pretendia obter?
4. **Documentação** — é possível identificar o que foi alterado e por quê?

In [ ]:
base_preparada = dados_originais.copy()

# Aplicações consideradas adequadas na base de referência.
base_preparada["canal_atendimento"] = canal_padronizado
base_preparada["valor_servicos"] = valor_numerico
base_preparada["ordens_por_hora_tecnica"] = ordens_por_hora.round(2)

print("Base preparada de referência")
display(base_preparada)

print("\nVerificações objetivas")
print("Unidade preservada:", base_preparada["unidade"].equals(dados_originais["unidade"]))
print("Tipo de serviço preservado:", base_preparada["tipo_servico"].equals(dados_originais["tipo_servico"]))
print("Categorias de canal após padronização:", sorted(base_preparada["canal_atendimento"].unique()))
print("Tipo de valor_servicos:", base_preparada["valor_servicos"].dtype)
print("Variável derivada criada:", "ordens_por_hora_tecnica" in base_preparada.columns)

## Parecer final

Completem no roteiro o registro de auditoria e respondam:

> **A base preparada pode ser liberada para as comparações previstas?**

O parecer deve mencionar pelo menos uma transformação aprovada, uma proposta rejeitada ou revista e uma evidência observada no notebook.

Lembrem-se: **não transformar** também pode ser a decisão correta quando a variável já atende à necessidade da análise.

## Critérios para avaliação da atividade

- Coerência das decisões com os objetivos da análise.
- Preservação do significado das informações necessárias.
- Uso das comparações antes/depois para verificar os resultados.
- Documentação clara da decisão, da justificativa e da evidência.

**Entrega:** roteiro preenchido por dupla, acompanhado da execução do notebook durante a atividade.